# Data Preprocessing — Exercises

Companion to the note [Data Preprocessing](Data%20Preprocessing.md).

Practise scaling (min–max, z-score, robust), missing-data mechanisms and imputation, outlier detection and robust losses, class imbalance (class weights, SMOTE, threshold moving, prior correction) and the pipeline rule: statistics from train only.

**16 exercises** · 🧠 Concept ×4 · ✍️ By hand ×5 · 💻 Code ×7

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from scipy.optimize import minimize
rng = np.random.default_rng(0)
x_small = np.array([2., 4., 6., 8., 20.])

## Part A · Concepts

### Exercise 1 · Who needs scaling?
*🧠 Concept · ★☆☆*

For each model say whether feature scaling matters and why: (a) $k$-NN, (b) a decision tree, (c) ridge regression, (d) PCA,
(e) unregularised OLS solved in closed form, (f) logistic regression trained by gradient descent.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Scaling matters when the algorithm uses distances, penalties on weight sizes, variances, or step sizes shared across features.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) **Yes**: Euclidean distance is dominated by large-scale features. (b) **No**: splits are thresholds on one feature, invariant to monotone transforms.
(c) **Yes**: $\lambda\lVert w\rVert^2$ penalises coefficients of small-scale features more. (d) **Yes**: PCA finds directions of maximal variance, so
large-unit features dominate. (e) **No** for predictions (the solution rescales its coefficients), though conditioning improves.
(f) **Yes**: one learning rate for badly-scaled features gives an elongated loss surface and slow, zig-zagging [[Gradient Descent]].

</details>

### Exercise 2 · MCAR, MAR or MNAR?
*🧠 Concept · ★★☆*

Classify the missingness mechanism and say whether dropping incomplete rows biases the analysis.

1. A lab robot randomly fails on 2 % of blood samples.
2. Older respondents more often skip the "income" question, and age is recorded.
3. People with very high income tend to skip the "income" question.
4. A sensor stops reporting when the temperature exceeds its range.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

MCAR: missingness independent of everything. MAR: depends only on observed variables. MNAR: depends on the missing value itself.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. **MCAR**: dropping rows loses data but does not bias.
2. **MAR**: missingness depends on observed age; dropping biases toward the young, but conditioning on age (model-based imputation) can fix it.
3. **MNAR**: depends on the unobserved income itself; no imputation using observed data fully fixes it. A missing-indicator feature at least lets the model use the signal.
4. **MNAR** (censoring): missing exactly when the value is extreme.

</details>

### Exercise 3 · Imbalance: what and where
*🧠 Concept · ★★☆*

A fraud dataset has 0.5 % positives. (a) Why is accuracy useless and why is PR-AUC preferred to ROC-AUC here?
(b) A colleague applies SMOTE to the whole dataset and then runs cross-validation. What is wrong?
(c) Contrast class weights, resampling and threshold moving: which change the model and which only change the decision?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

ROC's FPR has the huge negative class in its denominator. Synthetic points created from a validation sample leak it.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Predicting "no fraud" gives 99.5 % accuracy. ROC-AUC can look great while precision is terrible, because many false positives are a
tiny FPR; PR-AUC exposes precision directly ([[Model Evaluation and Metrics]]).
(b) Synthetic samples interpolated between points that end up in different folds leak validation information, and the validation folds
contain synthetic points, so the class ratio no longer matches reality. Resample **only the training fold**, inside the pipeline.
(c) Class weights and resampling change the training objective (and thus the model). Threshold moving keeps the model and only changes
the operating point on its scores.

</details>

### Exercise 4 · Outliers: error or signal?
*🧠 Concept · ★☆☆*

You find a house with 120 rooms and a patient with body temperature 45 °C. Describe how you would decide whether to remove, cap
(winsorize) or keep each, and when a robust loss (Huber) is preferable to deleting points.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

First ask whether the value is *possible*. Then ask whether the extreme points are the interesting ones.

</details>

<details>
<summary><b>✅ Solution</b></summary>

45 °C is physiologically impossible → a data-entry or unit error: fix or set to missing. 120 rooms may be a hotel mislabelled as a house
(a different population: remove or flag) or a typo for 12. Winsorizing keeps the row but limits its leverage. A robust loss is preferable
when you cannot inspect points individually, or when outliers are genuine but should not dominate the fit. In anomaly detection the
outliers *are* the signal ([[Anomaly Detection]]).

</details>

## Part B · By hand

### Exercise 5 · Three scalers on five numbers
*✍️ By hand · ★☆☆*

For $x = (2, 4, 6, 8, 20)$ compute (a) the min–max scaled value of $6$, (b) the z-score of $20$ using the population standard deviation,
(c) the robust-scaled value of $20$: $(x - \text{median})/\text{IQR}$ with $Q_1 = 4$, $Q_3 = 8$.

In [ ]:
minmax_6 = None
z_20 = None
robust_20 = None

check('min-max(6)', minmax_6, MinMaxScaler().fit_transform(x_small[:, None])[2, 0])
check('z(20)', z_20, StandardScaler().fit_transform(x_small[:, None])[4, 0])
check('robust(20)', robust_20, RobustScaler().fit_transform(x_small[:, None])[4, 0])

<details>
<summary><b>💡 Hint</b></summary>

$\mu = 8$, $\sigma = \sqrt{40}\approx6.325$, median $=6$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $(6-2)/(20-2) = 4/18 \approx 0.222$. (b) $\sigma^2 = (36+16+4+0+144)/5 = 40$, $z = 12/\sqrt{40}\approx 1.897$.
(c) $(20-6)/(8-4) = 3.5$. Note how the outlier squeezes min–max values of the other points into $[0, 0.33]$.

```python
minmax_6 = 4 / 18
z_20 = 12 / np.sqrt(40)
robust_20 = 3.5
```

</details>

### Exercise 6 · Masking: z-score vs. IQR rule
*✍️ By hand · ★★☆*

On $x=(2,4,6,8,20)$ apply (a) the rule "outlier if $|z|>3$" and (b) Tukey's rule "outlier if outside $[Q_1 - 1.5\,\text{IQR},\ Q_3 + 1.5\,\text{IQR}]$".
Store the fences of (b) and the number of points each rule flags. Why does the z-rule fail here?

In [ ]:
fences = None   # (lower, upper)
n_flag_z = None
n_flag_iqr = None

_q1, _q3 = np.percentile(x_small, [25, 75])
check('fences', fences, (_q1 - 1.5*(_q3-_q1), _q3 + 1.5*(_q3-_q1)))
check('flagged by z', n_flag_z, int(np.sum(np.abs(StandardScaler().fit_transform(x_small[:, None])) > 3)))
check('flagged by IQR', n_flag_iqr, int(np.sum((x_small < _q1 - 1.5*(_q3-_q1)) | (x_small > _q3 + 1.5*(_q3-_q1)))))

<details>
<summary><b>💡 Hint</b></summary>

IQR $= 4$, so the fences are $4 - 6$ and $8 + 6$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) The largest $|z|$ is $1.90$: **no** point is flagged. (b) Fences $(-2, 14)$: $20$ is flagged (1 point).
The outlier inflates both $\mu$ and $\sigma$, hiding itself (**masking**). Quantile-based rules are robust; with $n=5$ points, $|z|$ (population std) can never exceed
$\sqrt{n-1} = 2$, so the 3-sigma rule can never fire.

```python
fences = (-2.0, 14.0)
n_flag_z = 0
n_flag_iqr = 1
```

</details>

### Exercise 7 · Mean imputation shrinks variance
*✍️ By hand · ★★☆*

A feature has observed values $(1, 3, 5, 7)$ and 2 missing values. (a) Compute the population variance of the observed values.
(b) Impute the mean and recompute the variance over all 6 values. (c) Show in general that imputing $m$ of $n$ values by the observed
mean multiplies the population variance by $(n-m)/n$. Why does this matter for downstream models?

In [ ]:
var_obs = None
var_imputed = None

_o = np.array([1,3,5,7.])
check('observed variance', var_obs, _o.var())
check('after mean imputation', var_imputed, SimpleImputer().fit_transform(np.r_[_o, np.nan, np.nan][:, None]).var())

<details>
<summary><b>💡 Hint</b></summary>

Imputed values sit exactly at the mean, so they add 0 to the sum of squared deviations but count in $n$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Mean $4$, squared deviations $9+1+1+9 = 20$, variance $20/4 = 5$. (b) The sum stays $20$ but $n=6$: $20/6\approx3.33$.
(c) $\frac1n\sum_{\text{all}} = \frac1n\cdot (n-m)\,\hat\sigma^2_{\text{obs}}$. Mean imputation understates variability and weakens correlations
with other features; add a missing-indicator or use model/kNN imputation when missingness is substantial.

```python
var_obs = 5.0
var_imputed = 20 / 6
```

</details>

### Exercise 8 · Balanced class weights
*✍️ By hand · ★☆☆*

With 900 negatives and 100 positives, sklearn's `class_weight='balanced'` uses $w_k = \dfrac{N}{K\,N_k}$. Compute both weights and show
that afterwards both classes contribute equal total weight to the loss.

In [ ]:
w_neg = None
w_pos = None

_cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=np.r_[np.zeros(900), np.ones(100)])
check('w_neg', w_neg, _cw[0]); check('w_pos', w_pos, _cw[1])

<details>
<summary><b>💡 Hint</b></summary>

$N = 1000$, $K = 2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$w_0 = 1000/(2\cdot900)\approx0.556$, $w_1 = 1000/(2\cdot100) = 5$. Total weights: $900\cdot0.556 = 500 = 100\cdot5$.
Weighting is equivalent in expectation to resampling the classes to a 50/50 ratio, without duplicating data.

```python
w_neg = 1000 / 1800
w_pos = 5.0
```

</details>

### Exercise 9 · Undo the resampling: prior correction
*✍️ By hand · ★★★*

You trained a calibrated classifier on a **re-balanced** set (50 % positives), but in production only 10 % are positive.
For an input where the model outputs $p_s = 0.7$, what is the correct production probability?
Derive the formula from Bayes' rule assuming $p(x\mid y)$ is unchanged.

In [ ]:
p_corrected = None

# brute force: simulate class-conditional likelihood ratio implied by p_s under the balanced prior
_lr = 0.7 / 0.3          # p(x|1)/p(x|0) since the training prior was 50/50
check('corrected probability', p_corrected, _lr * 0.1 / (_lr * 0.1 + 0.9))

<details>
<summary><b>💡 Hint</b></summary>

$p(y\mid x)\propto p(x\mid y)\,\pi_y$. Changing the prior $\pi_y$ re-weights the posterior by $\pi_y^{\text{new}}/\pi_y^{\text{train}}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$p(y=1\mid x) = \dfrac{p_s\,\pi_1/\pi_1^s}{p_s\,\pi_1/\pi_1^s + (1-p_s)\,\pi_0/\pi_0^s} = \dfrac{0.7\cdot0.2}{0.7\cdot0.2 + 0.3\cdot1.8} = \dfrac{0.14}{0.68}\approx0.206$.
Resampling distorts probabilities; if you need calibrated outputs, correct them (or use the equivalent shifted threshold).

```python
p_corrected = 0.14 / 0.68
```

</details>

## Part C · Code

### Exercise 10 · Three scalers from scratch
*💻 Code · ★☆☆*

Implement `fit_scaler(X, kind)` returning `(center, scale)` for `kind` in `'minmax'`, `'standard'`, `'robust'`, and
`transform(X, center, scale)`. Check against sklearn on a skewed random matrix.

In [ ]:
def fit_scaler(X, kind):
    return None

def transform(X, center, scale):
    return (X - center) / scale

X_sk = rng.lognormal(size=(100, 3))
out = {k: (None if fit_scaler(X_sk, k) is None else transform(X_sk, *fit_scaler(X_sk, k))) for k in ['minmax', 'standard', 'robust']}

check('min-max', out['minmax'], MinMaxScaler().fit_transform(X_sk))
check('standard', out['standard'], StandardScaler().fit_transform(X_sk))
check('robust', out['robust'], RobustScaler().fit_transform(X_sk))

<details>
<summary><b>💡 Hint</b></summary>

Min–max: center = min, scale = max − min. Robust: center = median, scale = $Q_3-Q_1$ via `np.percentile`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

All three are affine maps $x' = (x - c)/s$; they differ only in how $c$ and $s$ are estimated. Only the robust one ignores the long right tail of the lognormal.

```python
def fit_scaler(X, kind):
    if kind == 'minmax':
        return X.min(0), X.max(0) - X.min(0)
    if kind == 'standard':
        return X.mean(0), X.std(0)
    if kind == 'robust':
        q1, q3 = np.percentile(X, [25, 75], axis=0)
        return np.median(X, 0), q3 - q1
    raise ValueError(kind)

def transform(X, center, scale):
    return (X - center) / scale

X_sk = rng.lognormal(size=(100, 3))
out = {k: transform(X_sk, *fit_scaler(X_sk, k)) for k in ['minmax', 'standard', 'robust']}
```

</details>

### Exercise 11 · Scaling and k-NN
*💻 Code · ★☆☆*

On `load_breast_cancer` (75/25 stratified split, `random_state=0`) compare 5-NN test accuracy on raw features (`acc_raw`) and on
features standardised with **train** statistics (`acc_scaled`). Print the per-feature standard deviations to see why.

In [ ]:
X_bc, y_bc = load_breast_cancer(return_X_y=True)
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(X_bc, y_bc, test_size=0.25, random_state=0, stratify=y_bc)
acc_raw = None
acc_scaled = None

check('raw', acc_raw, KNeighborsClassifier(5).fit(Xb_tr, yb_tr).score(Xb_te, yb_te))
_s = StandardScaler().fit(Xb_tr)
check('scaled', acc_scaled, KNeighborsClassifier(5).fit(_s.transform(Xb_tr), yb_tr).score(_s.transform(Xb_te), yb_te))

<details>
<summary><b>💡 Hint</b></summary>

Feature standard deviations range from about 0.003 to over 500.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Raw ≈ 0.92, scaled ≈ 0.95. Without scaling, "worst area" (std ≈ 570) dominates every distance and the other 29 features are almost ignored.

```python
X_bc, y_bc = load_breast_cancer(return_X_y=True)
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(X_bc, y_bc, test_size=0.25, random_state=0, stratify=y_bc)
print(X_bc.std(0).min().round(4), X_bc.std(0).max().round(1))
acc_raw = KNeighborsClassifier(5).fit(Xb_tr, yb_tr).score(Xb_te, yb_te)
mu, sd = Xb_tr.mean(0), Xb_tr.std(0)
acc_scaled = KNeighborsClassifier(5).fit((Xb_tr - mu) / sd, yb_tr).score((Xb_te - mu) / sd, yb_te)
print(acc_raw, acc_scaled)
```

</details>

### Exercise 12 · Mean imputation with a missing indicator
*💻 Code · ★★☆*

Implement `impute_with_indicator(X_train, X_apply)` that replaces NaNs in `X_apply` with the **train** column means and appends one 0/1
indicator column for every column that had missing values **in the training data** (same as `SimpleImputer(add_indicator=True)`).

In [ ]:
X_mis_tr = rng.normal(size=(50, 3)); X_mis_tr[rng.random((50, 3)) < [0.2, 0.0, 0.1]] = np.nan
X_mis_te = rng.normal(size=(20, 3)); X_mis_te[rng.random((20, 3)) < [0.2, 0.0, 0.1]] = np.nan

def impute_with_indicator(X_train, X_apply):
    return None

imp_te = impute_with_indicator(X_mis_tr, X_mis_te)

check('matches SimpleImputer(add_indicator=True)', imp_te, SimpleImputer(add_indicator=True).fit(X_mis_tr).transform(X_mis_te))

<details>
<summary><b>💡 Hint</b></summary>

`np.nanmean(X_train, 0)`; the indicator columns are `np.isnan(X_apply)[:, cols]` with `cols = np.where(np.isnan(X_train).any(0))[0]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The indicator lets the model learn that "missing" itself is informative (important for MNAR). Column 2 has no missing values in training, so
it gets no indicator, even if it had NaNs at test time.

```python
X_mis_tr = rng.normal(size=(50, 3)); X_mis_tr[rng.random((50, 3)) < [0.2, 0.0, 0.1]] = np.nan
X_mis_te = rng.normal(size=(20, 3)); X_mis_te[rng.random((20, 3)) < [0.2, 0.0, 0.1]] = np.nan

def impute_with_indicator(X_train, X_apply):
    mu = np.nanmean(X_train, 0)
    miss = np.isnan(X_apply)
    filled = np.where(miss, mu, X_apply)
    cols = np.where(np.isnan(X_train).any(0))[0]
    return np.c_[filled, miss[:, cols].astype(float)]

imp_te = impute_with_indicator(X_mis_tr, X_mis_te)
```

</details>

### Exercise 13 · kNN imputation beats the mean
*💻 Code · ★★★*

Column 0 of `X_corr` is strongly correlated with columns 1–2; 30 % of column 0 is missing (`mask`). Implement
`knn_impute_col0(X, k)`: for each row with a missing column 0, find the $k$ nearest rows **with column 0 observed** using Euclidean distance on
columns 1–2, and fill in the mean of their column-0 values. Compare RMSE against the true values for kNN (`rmse_knn`) and mean imputation (`rmse_mean`).

In [ ]:
n = 300
z = rng.normal(size=n)
X_true = np.c_[z + 0.1 * rng.normal(size=n), z + 0.3 * rng.normal(size=n), -z + 0.3 * rng.normal(size=n)]
mask = rng.random(n) < 0.3
X_corr = X_true.copy(); X_corr[mask, 0] = np.nan

def knn_impute_col0(X, k=5):
    return None

X_knn = knn_impute_col0(X_corr, k=5)
rmse_knn = None
rmse_mean = None

check('matches KNNImputer', X_knn, KNNImputer(n_neighbors=5).fit_transform(X_corr))
check('kNN much better than mean', None if rmse_knn is None else rmse_knn < rmse_mean / 2, True)

<details>
<summary><b>💡 Hint</b></summary>

Donors: `obs = ~np.isnan(X[:, 0])`. Distances from each missing row to every donor on columns 1–2; take `argsort(...)[:, :k]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

kNN imputation exploits the correlation and gets RMSE ≈ 0.25 versus ≈ 0.86 for the mean. With only one column missing, sklearn's `nan_euclidean`
distance is a constant multiple of the Euclidean distance on the other columns, so the result matches `KNNImputer` exactly.
Remember to fit the imputer on training data only.

```python
n = 300
z = rng.normal(size=n)
X_true = np.c_[z + 0.1 * rng.normal(size=n), z + 0.3 * rng.normal(size=n), -z + 0.3 * rng.normal(size=n)]
mask = rng.random(n) < 0.3
X_corr = X_true.copy(); X_corr[mask, 0] = np.nan

def knn_impute_col0(X, k=5):
    X = X.copy()
    miss = np.isnan(X[:, 0])
    donors = X[~miss]
    D = ((X[miss][:, None, 1:] - donors[None, :, 1:]) ** 2).sum(-1)
    idx = np.argsort(D, axis=1, kind='stable')[:, :k]
    X[miss, 0] = donors[idx, 0].mean(1)
    return X

X_knn = knn_impute_col0(X_corr, k=5)
rmse_knn = np.sqrt(np.mean((X_knn[mask, 0] - X_true[mask, 0]) ** 2))
rmse_mean = np.sqrt(np.mean((np.nanmean(X_corr[:, 0]) - X_true[mask, 0]) ** 2))
print(rmse_knn, rmse_mean)
```

</details>

### Exercise 14 · Robust regression with the Huber loss
*💻 Code · ★★★*

The data below follow $y = 1 + 2x$ but 10 % of targets are corrupted by $+15$. Implement Huber regression by **iteratively reweighted least squares**:
residual $r_i$, weight $w_i = 1$ if $|r_i|\le\delta$ else $\delta/|r_i|$, then solve weighted least squares; repeat. Use $\delta = 1$ and 100 iterations.
Compare the slope with OLS.

In [ ]:
x_hu = rng.uniform(0, 10, 100)
y_hu = 1 + 2 * x_hu + 0.5 * rng.normal(size=100)
y_hu[rng.random(100) < 0.1] += 15

def huber_irls(x, y, delta=1.0, iters=100):
    # TODO: return np.array([intercept, slope])
    return None

theta_huber = huber_irls(x_hu, y_hu)

def _huber_obj(t):
    r = y_hu - t[0] - t[1] * x_hu
    a = np.abs(r)
    return np.sum(np.where(a <= 1.0, 0.5 * r ** 2, 1.0 * (a - 0.5)))
_ref = minimize(_huber_obj, np.array([0.0, 0.0]), method='Nelder-Mead', options=dict(xatol=1e-10, fatol=1e-12, maxiter=20000)).x
check('Huber solution', theta_huber, _ref, tol=1e-3)

<details>
<summary><b>💡 Hint 1</b></summary>

Weighted least squares: `A = np.c_[np.ones_like(x), x]`, solve $(A^\top W A)\theta = A^\top W y$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Start from the OLS solution; use `np.maximum(np.abs(r), 1e-12)` to avoid division by zero.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The Huber loss $\rho(r)=\tfrac12r^2$ for $|r|\le\delta$ and $\delta(|r|-\tfrac\delta2)$ otherwise is quadratic for small and linear for large residuals.
IRLS solves its stationarity condition $\sum_i \psi(r_i)\,a_i = 0$ with $\psi(r) = w(r)\,r$. OLS is pulled toward the outliers (≈ $(2.56, 2.24)$),
while Huber stays close to the truth (≈ $(1.03, 2.04)$).

```python
x_hu = rng.uniform(0, 10, 100)
y_hu = 1 + 2 * x_hu + 0.5 * rng.normal(size=100)
y_hu[rng.random(100) < 0.1] += 15

def huber_irls(x, y, delta=1.0, iters=100):
    A = np.c_[np.ones_like(x), x]
    t = np.linalg.lstsq(A, y, rcond=None)[0]
    for _ in range(iters):
        r = y - A @ t
        w = np.where(np.abs(r) <= delta, 1.0, delta / np.maximum(np.abs(r), 1e-12))
        t = np.linalg.solve(A.T @ (w[:, None] * A), A.T @ (w * y))
    return t

theta_huber = huber_irls(x_hu, y_hu)
print('OLS  ', np.linalg.lstsq(np.c_[np.ones(100), x_hu], y_hu, rcond=None)[0].round(3))
print('Huber', theta_huber.round(3))
```

</details>

### Exercise 15 · SMOTE from scratch
*💻 Code · ★★★*

Implement `smote(X_min, n_new, k, rng)`: for each new sample pick a random minority point $x_i$, one of its $k$ nearest minority neighbours $x_{nn}$
(excluding itself) and a random $u\sim U(0,1)$, and output $x_i + u\,(x_{nn}-x_i)$. Generate 100 synthetic points from 40 minority points.

In [ ]:
X_min = rng.normal(size=(40, 2)) * [1, 0.3] + [3, 1]

def smote(X_min, n_new, k=5, rng=None):
    return None

X_syn = smote(X_min, 100, k=5, rng=np.random.default_rng(1))

def _on_knn_segment(s, X, k=5):
    D = ((X[:, None] - X[None]) ** 2).sum(-1); np.fill_diagonal(D, np.inf)
    nn = np.argsort(D, 1)[:, :k]
    for i in range(len(X)):
        for j in nn[i]:
            d = X[j] - X[i]; u = (s - X[i]) @ d / (d @ d)
            if -1e-9 <= u <= 1 + 1e-9 and np.allclose(X[i] + u * d, s, atol=1e-8):
                return True
    return False
if X_syn is not None:
    check('shape', np.shape(X_syn), (100, 2))
    check('every synthetic point lies on a segment to a k-NN', all(_on_knn_segment(s, X_min) for s in X_syn), True)
else:
    check('shape', None, (100, 2))

<details>
<summary><b>💡 Hint</b></summary>

Precompute the neighbour lists once: distance matrix with `np.inf` on the diagonal, then `np.argsort(D, 1)[:, :k]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Unlike random oversampling (exact duplicates, which encourage overfitting), SMOTE creates new points along line segments inside the minority region.
It can also create points in regions belonging to the other class when classes overlap, and it must be applied **inside** the training folds.

```python
X_min = rng.normal(size=(40, 2)) * [1, 0.3] + [3, 1]

def smote(X_min, n_new, k=5, rng=None):
    rng = rng or np.random.default_rng()
    D = ((X_min[:, None] - X_min[None]) ** 2).sum(-1)
    np.fill_diagonal(D, np.inf)
    nn = np.argsort(D, 1)[:, :k]
    i = rng.integers(len(X_min), size=n_new)
    j = nn[i, rng.integers(k, size=n_new)]
    u = rng.random((n_new, 1))
    return X_min[i] + u * (X_min[j] - X_min[i])

X_syn = smote(X_min, 100, k=5, rng=np.random.default_rng(1))
```

</details>

### Exercise 16 · Threshold moving
*💻 Code · ★★☆*

On an imbalanced dataset (5 % positives), fit logistic regression on the training part, then on a **validation** split choose the threshold
$t\in\{0.01, 0.02,\dots,0.99\}$ maximising F1. Store it in `t_best`, and the validation F1 at $0.5$ and at `t_best` in `f1_05`, `f1_best`.

In [ ]:
X_im, y_im = make_classification(n_samples=4000, weights=[0.95], flip_y=0.01, class_sep=0.8, random_state=0)
Xi_tr, Xi_va, yi_tr, yi_va = train_test_split(X_im, y_im, test_size=0.5, random_state=0, stratify=y_im)
p_va = LogisticRegression().fit(Xi_tr, yi_tr).predict_proba(Xi_va)[:, 1]
t_best = None
f1_05 = None
f1_best = None

_ts = np.round(np.arange(0.01, 1.0, 0.01), 2)
_f = [f1_score(yi_va, p_va >= t) for t in _ts]
check('best threshold', t_best, _ts[int(np.argmax(_f))])
check('F1 at best >= F1 at 0.5', None if f1_best is None else f1_best >= f1_05, True)

<details>
<summary><b>💡 Hint</b></summary>

Loop over `np.round(np.arange(0.01, 1.0, 0.01), 2)` and compute `f1_score(yi_va, p_va >= t)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Here the F1-optimal threshold is 0.15, roughly doubling F1 (0.17 → 0.36): it lies well below 0.5 because positives are rare and recall is cheap to buy. Choosing it on **validation** data (not test)
keeps the final evaluation honest. The model is unchanged; only the decision rule moved.

```python
X_im, y_im = make_classification(n_samples=4000, weights=[0.95], flip_y=0.01, class_sep=0.8, random_state=0)
Xi_tr, Xi_va, yi_tr, yi_va = train_test_split(X_im, y_im, test_size=0.5, random_state=0, stratify=y_im)
p_va = LogisticRegression().fit(Xi_tr, yi_tr).predict_proba(Xi_va)[:, 1]
ts = np.round(np.arange(0.01, 1.0, 0.01), 2)
f1s = np.array([f1_score(yi_va, p_va >= t) for t in ts])
t_best = ts[int(np.argmax(f1s))]
f1_05 = f1_score(yi_va, p_va >= 0.5)
f1_best = f1s.max()
print(t_best, f1_05, f1_best)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Data Preprocessing](Data%20Preprocessing.md).*